# Sentiment Analysis using NLP Pipeline & Machine Learning

# Load Dataset and Import Lib

In [24]:
# Import Libraries
import pandas as pd
import numpy as np

# Load Dataset
df = pd.read_csv("sentimentdataset.csv")
df.head()

,Unnamed: 0.1,Unnamed: 0,Text,Sentiment,Timestamp,User,Platform,Hashtags,Retweets,Likes,Country,Year,Month,Day,Hour
0,0,0,Enjoying a beautiful day at the park! ...,Positive,2023-01-15 12:30:00,User123,Twitter,#Nature #Park,15.0,30.0,USA,2023,1,15,12
1,1,1,Traffic was terrible this morning. ...,Negative,2023-01-15 08:45:00,CommuterX,Twitter,#Traffic #Morning,5.0,10.0,Canada,2023,1,15,8
2,2,2,Just finished an amazing workout! 💪 ...,Positive,2023-01-15 15:45:00,FitnessFan,Instagram,#Fitness #Workout,20.0,40.0,USA,2023,1,15,15
3,3,3,Excited about the upcoming weekend getaway! ...,Positive,2023-01-15 18:20:00,AdventureX,Facebook,#Travel #Adventure,8.0,15.0,UK,2023,1,15,18
4,4,4,Trying out a new recipe for dinner tonight. ...,Neutral,2023-01-15 19:55:00,ChefCook,Instagram,#Cooking #Food,12.0,25.0,Australia,2023,1,15,19


In [25]:
df.shape

(732, 15)

In [26]:
df.columns

Index(['Unnamed: 0.1', 'Unnamed: 0', 'Text', 'Sentiment', 'Timestamp', 'User',
       'Platform', 'Hashtags', 'Retweets', 'Likes', 'Country', 'Year', 'Month',
       'Day', 'Hour'],
      dtype='object')

In [27]:
df.isnull().sum()

Unnamed: 0.1    0
Unnamed: 0      0
Text            0
Sentiment       0
Timestamp       0
User            0
Platform        0
Hashtags        0
Retweets        0
Likes           0
Country         0
Year            0
Month           0
Day             0
Hour            0
dtype: int64

In [28]:
print(df['Sentiment'].value_counts())

Sentiment
Positive               44
Joy                    42
Excitement             32
Happy                  14
Neutral                14
                       ..
Vibrancy                1
Culinary Adventure      1
Mesmerizing             1
Thrilling Journey       1
Winter Magic            1
Name: count, Length: 279, dtype: int64


In [29]:
# Handle Labels

In [30]:
def map_sentiment(label):
    label = str(label).lower()

    if any(word in label for word in ['positive','joy','happy','excited','love']):
        return "positive"
    elif any(word in label for word in  ['negative','sad','anger','hate']):
        return "negative"
    else:
        return "neutral"

In [31]:
df['sentiment'] = df['Sentiment'].apply(map_sentiment)

In [32]:
df['sentiment'].value_counts()

sentiment
neutral     595
positive    114
negative     23
Name: count, dtype: int64

# NLP Preprocessing

In [33]:
# Import NLP libraries
import re
import nltk

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

nltk.download("stopwords")
nltk.download("wordnet")

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\PiyushShaileshBorse\AppData\Roaming\nltk_data
[nltk_data]     ...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\PiyushShaileshBorse\AppData\Roaming\nltk_data
[nltk_data]     ...
[nltk_data]   Package wordnet is already up-to-date!


True

### Initialize tools

In [34]:
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

### Preprocess function

In [35]:

def preprocess_text(text):
    text = text.lower()
    text = re.sub(r'http\S+|www\S+','',text)
    text = re.sub(r'<.*?>','',text)
    text = re.sub(r'[^a-zA-Z]','',text)

    words = text.split()
    words = [w for w in words if w not in stop_words]
    words = [lemmatizer.lemmatize(w) for w in words]

    return ''.join(words)

In [36]:
# Apply preprocessing
df['clean_text'] = df['Text'].apply(preprocess_text)

### Compare before vs after

In [37]:
for i in range(2):
    print("Original:" , df['Text'][i])
    print("Cleaned:" , df['clean_text'][i])
    print("-"*40)

Original:  Enjoying a beautiful day at the park!              
Cleaned: enjoyingabeautifuldayatthepark
----------------------------------------
Original:  Traffic was terrible this morning.                 
Cleaned: trafficwasterriblethismorning
----------------------------------------


# Feature Engineering

### Define X and Y

In [38]:
x = df['clean_text']
y = df['sentiment']

### Import train-split , Split data

In [39]:
from sklearn.model_selection import train_test_split

### split data

In [40]:
x_train,x_test,y_train,y_test = train_test_split(x,y,test_size=0.2,random_state=21)

# Vectorization

In [41]:
# import vector 
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

### BOW

In [42]:
bow = CountVectorizer(max_features=5000)

x_train_bow = bow.fit_transform(x_train)
x_test_bow = bow.transform(x_test)

### Tf-IDF

In [43]:
tfidf = CountVectorizer(max_features=5000)

x_train_tfidf = tfidf.fit_transform(x_train)
x_test_tfidf = tfidf.transform(x_test)

# Model Building

In [44]:
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.tree import DecisionTreeClassifier

### logistic Regression

In [45]:
lr = LogisticRegression(max_iter=200)
lr.fit(x_train_tfidf, y_train)

y_pred_lr = lr.predict(x_test_tfidf)

### Naive Bayes 

In [46]:
nb = MultinomialNB()
nb.fit(x_train_tfidf, y_train)

y_pred_nb = nb.predict(x_test_tfidf)

### Decision Tree

In [47]:
dt = DecisionTreeClassifier()
dt.fit(x_train_tfidf, y_train)

y_pred_dt = dt.predict(x_test_tfidf)

# Evalution

In [48]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

### Evalution funtion

In [49]:
def evaluate(y_true, y_pred):
    return (
        accuracy_score(y_true, y_pred),
        precision_score(y_true, y_pred, average='weighted'),
        recall_score(y_true, y_pred, average='weighted'),
        f1_score(y_true, y_pred, average='weighted')
    )

### Compare model

In [50]:
print("LR:", evaluate(y_test, y_pred_lr))
print("NB:", evaluate(y_test, y_pred_nb))
print("DT:", evaluate(y_test, y_pred_dt))

LR: (0.8571428571428571, 0.7346938775510204, 0.8571428571428571, 0.7912087912087913)
NB: (0.8571428571428571, 0.7346938775510204, 0.8571428571428571, 0.7912087912087913)
DT: (0.8571428571428571, 0.7346938775510204, 0.8571428571428571, 0.7912087912087913)


C:\anaconda\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\anaconda\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\anaconda\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


# Final Result

In [51]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred_lr))

              precision    recall  f1-score   support

    negative       0.00      0.00      0.00         5
     neutral       0.86      1.00      0.92       126
    positive       0.00      0.00      0.00        16

    accuracy                           0.86       147
   macro avg       0.29      0.33      0.31       147
weighted avg       0.73      0.86      0.79       147



C:\anaconda\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\anaconda\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\anaconda\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
